# SAM Fine-tuning on Indian Faces - Kaggle

Fine-tunes the SAM encoder on Indian face images for age-accurate face transformation.

**Strategy:** Freeze StyleGAN2 decoder (preserves generation quality), only train the encoder.

## Datasets
| Dataset | Indian Faces | Resolution | Age Labels |
|---------|-------------|------------|------------|
| FairFace | ~15,000 | 448x448 | Age ranges |
| UTKFace | ~3,000 | 200x200 | Exact ages |
| **Combined** | **~18,000** | mixed | yes |

## Before Running
1. **Add Data** > Upload your `agevision-sam-training-dataset` (SAM code + `sam_ffhq_aging.pt`)
2. **Add Data** > Search `fairface` > Add **mehmoodsheikh/fairface-dataset**
3. **Add Data** > Search `utkface` > Add **jangedoo/utkface-new**
4. **Settings** > Accelerator > **GPU T4 x2**


In [ ]:
#@title Step 1: Setup & GPU Check
import os, sys, shutil, zipfile, glob, re, csv, time, gc, logging
import torch

# ---- GPU ----
print(f'PyTorch: {torch.__version__}')
if not torch.cuda.is_available():
    raise RuntimeError('No GPU! Settings > Accelerator > GPU T4 x2')
for i in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(i)
    mem = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f'GPU {i}: {name} ({mem:.1f} GB)')

# ---- Locate SAM training package ----
WORK_DIR = '/kaggle/working/agevision_sam'
os.makedirs(WORK_DIR, exist_ok=True)
SAM_DIR = f'{WORK_DIR}/sam'
CHECKPOINTS_DIR = f'{WORK_DIR}/checkpoints'

# Search for SAM code in any input dataset
def find_in_input(name, is_dir=True):
    for pattern in [f'/kaggle/input/*/{name}', f'/kaggle/input/*/*/{name}',
                    f'/kaggle/input/*/*/*/{name}']:
        for p in glob.glob(pattern):
            if (is_dir and os.path.isdir(p)) or (not is_dir and os.path.isfile(p)):
                return p
    return None

# Try direct dataset first, then zip fallback
src_sam = find_in_input('sam')
src_ckpt = find_in_input('checkpoints')

if src_sam:
    if not os.path.isdir(SAM_DIR):
        shutil.copytree(src_sam, SAM_DIR)
    print(f'SAM code: {src_sam}')
else:
    # Try zip
    zip_path = find_in_input('colab_sam_training.zip', is_dir=False)
    if zip_path:
        print(f'Extracting {zip_path}...')
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(WORK_DIR)
        print('Extracted.')
    else:
        raise FileNotFoundError('SAM training package not found! Add your dataset via Add Data.')

if src_ckpt and not os.path.isdir(CHECKPOINTS_DIR):
    print('Copying checkpoints (1-2 min)...')
    shutil.copytree(src_ckpt, CHECKPOINTS_DIR)
    print('Done.')

# ---- Python package structure ----
API_DIR = f'{WORK_DIR}/agevision_api'
os.makedirs(API_DIR, exist_ok=True)
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)

if not os.path.exists(f'{API_DIR}/sam'):
    os.symlink(SAM_DIR, f'{API_DIR}/sam')

for d in [API_DIR, SAM_DIR, f'{SAM_DIR}/configs', f'{SAM_DIR}/datasets',
          f'{SAM_DIR}/models', f'{SAM_DIR}/models/stylegan2',
          f'{SAM_DIR}/models/stylegan2/op', f'{SAM_DIR}/models/encoders',
          f'{SAM_DIR}/scripts', f'{SAM_DIR}/utils']:
    if os.path.isdir(d):
        init = os.path.join(d, '__init__.py')
        if not os.path.exists(init):
            open(init, 'w').close()

if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

# ---- Write fresh train_config.py ----
with open(f'{SAM_DIR}/train_config.py', 'w') as f:
    f.write('''from dataclasses import dataclass

@dataclass
class SAMTrainConfig:
    checkpoint_path: str = ''
    output_dir: str = 'checkpoints/sam_finetune'
    data_root: str = ''
    age_csv: str = ''
    input_nc: int = 4
    output_size: int = 1024
    start_from_latent_avg: bool = False
    start_from_encoded_w_plus: bool = True
    batch_size: int = 2
    num_epochs: int = 20
    lr_encoder: float = 5e-5
    lr_decoder: float = 1e-5
    lr_discriminator: float = 1e-4
    weight_decay: float = 1e-5
    save_every: int = 5
    log_every: int = 10
    lambda_l1: float = 1.0
    lambda_identity: float = 1.0
    lambda_age: float = 0.1
    lambda_adv: float = 0.01
    lambda_lpips: float = 1.0
    img_size: int = 256
    num_workers: int = 2
    pin_memory: bool = True
    age_bins: int = 101
    freeze_decoder: bool = True
    gradient_accumulation_steps: int = 4
    gradient_checkpointing: bool = True
    device: str = 'cuda'
    mixed_precision: bool = True
''')
print('train_config.py written')

# ---- Write paths_config.py ----
with open(f'{SAM_DIR}/configs/paths_config.py', 'w') as f:
    f.write(f"""model_paths = {{
    'sam_ffhq_aging': '{CHECKPOINTS_DIR}/sam_ffhq_aging.pt',
    'sam_indian': '{CHECKPOINTS_DIR}/sam_indian_best.pt',
    'ir_se50': '{CHECKPOINTS_DIR}/model_ir_se50.pth',
    'stylegan_ffhq': '{CHECKPOINTS_DIR}/stylegan2-ffhq-config-f.pt',
    'pretrained_psp': '{CHECKPOINTS_DIR}/psp_ffhq_encode.pt',
    'age_predictor': '{CHECKPOINTS_DIR}/dex_age_classifier.pth',
}}\n""")
print('paths_config.py written')

# ---- Verify checkpoint ----
CHECKPOINT_PATH = f'{CHECKPOINTS_DIR}/sam_ffhq_aging.pt'
if not os.path.isfile(CHECKPOINT_PATH):
    # Fallback: find any .pt file
    for pt in glob.glob(f'{CHECKPOINTS_DIR}/*.pt'):
        CHECKPOINT_PATH = pt
        break
    else:
        raise FileNotFoundError('No checkpoint found! Make sure sam_ffhq_aging.pt is in your dataset.')

print(f'Checkpoint: {os.path.basename(CHECKPOINT_PATH)} ({os.path.getsize(CHECKPOINT_PATH)/1e9:.2f} GB)')
print('\n--- Step 1 Complete ---')


In [ ]:
#@title Step 2: Build Indian Face Dataset (FairFace + UTKFace)

DATASET_DIR = f'{WORK_DIR}/dataset'
IMAGES_DIR = f'{DATASET_DIR}/images'
os.makedirs(IMAGES_DIR, exist_ok=True)

MIN_AGE, MAX_AGE = 10, 80
indian_samples = []

# ============================================================
# 1. FairFace (primary — high quality, 448x448)
# ============================================================
print('='*55)
print('  [1/2] Searching for FairFace dataset...')
print('='*55)

FAIRFACE_AGE_MAP = {
    '0-2': 1, '3-9': 6, '10-19': 15, '20-29': 25,
    '30-39': 35, '40-49': 45, '50-59': 55, '60-69': 65,
    'more than 70': 75,
}

# Recursive search — finds CSVs at ANY depth under /kaggle/input/
ff_csvs = []
for root, dirs, files in os.walk('/kaggle/input'):
    for fname in files:
        if fname.startswith('fairface_label') and fname.endswith('.csv'):
            ff_csvs.append(os.path.join(root, fname))

print(f'  Found {len(ff_csvs)} FairFace CSV file(s)')

ff_count = 0
for csv_file in ff_csvs:
    csv_dir = os.path.dirname(csv_file)
    print(f'  Processing: {csv_file}')
    with open(csv_file, 'r', newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row.get('race', '').strip() != 'Indian':
                continue
            age = FAIRFACE_AGE_MAP.get(row.get('age', '').strip())
            if age is None or age < MIN_AGE or age > MAX_AGE:
                continue
            # Try multiple paths to find the image
            rel_path = row.get('file', '').strip()
            img_path = None
            for base in [csv_dir, os.path.dirname(csv_dir),
                         os.path.dirname(os.path.dirname(csv_dir))]:
                candidate = os.path.join(base, rel_path)
                if os.path.isfile(candidate):
                    img_path = candidate
                    break
            if img_path is None:
                continue
            indian_samples.append({
                'src': img_path,
                'filename': f'ff_{ff_count:06d}.jpg',
                'age': age,
                'gender': row.get('gender', 'Unknown').strip(),
            })
            ff_count += 1

print(f'  FairFace Indian faces: {ff_count}')
if ff_count == 0:
    print('  WARNING: No Indian faces extracted from FairFace.')
    print('  Check that the dataset has fairface_label_train.csv with race/age columns.')
    # Debug: show what files are in the dataset
    print('  Files found near CSV locations:')
    for c in ff_csvs[:2]:
        d = os.path.dirname(c)
        print(f'    {d}: {os.listdir(d)[:10]}')
    if not ff_csvs:
        print('  No FairFace CSV found anywhere in /kaggle/input/')
        print('  Add Data > search "fairface" > add a FairFace dataset')

# ============================================================
# 2. UTKFace (supplement — exact ages)
# ============================================================
print()
print('='*55)
print('  [2/2] Searching for UTKFace dataset...')
print('='*55)

utk_pattern = re.compile(r'^(\d+)_(\d)_(\d)_(\d+)')
utk_count = 0

# Also use os.walk for UTKFace (catches any depth)
for root, dirs, files in os.walk('/kaggle/input'):
    for fname in files:
        match = utk_pattern.match(fname)
        if not match or not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        age, gender, race = int(match.group(1)), int(match.group(2)), int(match.group(3))
        if race == 3 and MIN_AGE <= age <= MAX_AGE:
            indian_samples.append({
                'src': os.path.join(root, fname),
                'filename': f'utk_{utk_count:06d}.jpg',
                'age': age,
                'gender': 'Male' if gender == 0 else 'Female',
            })
            utk_count += 1

print(f'  UTKFace Indian faces: {utk_count}')
if utk_count == 0:
    print('  Not found. Add Data > search "utkface" > add jangedoo/utkface-new')

# ============================================================
# 3. Combine & Save
# ============================================================
total = len(indian_samples)
print(f'\n{"="*55}')
print(f'  TOTAL: {total} Indian face images')
print(f'    FairFace: {ff_count} | UTKFace: {utk_count}')
print(f'{"="*55}')

if total == 0:
    raise ValueError('No Indian faces found! Add FairFace and/or UTKFace datasets.')

print(f'\nCopying images to {IMAGES_DIR}...')
csv_path = f'{DATASET_DIR}/ages.csv'
copied = 0
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['filename', 'age', 'gender'])
    writer.writeheader()
    for s in indian_samples:
        dest = f'{IMAGES_DIR}/{s["filename"]}'
        if not os.path.exists(dest):
            shutil.copy2(s['src'], dest)
            copied += 1
        writer.writerow({'filename': s['filename'], 'age': s['age'], 'gender': s['gender']})

print(f'Copied {copied} images')

# Age distribution
print('\nAge Distribution:')
age_dist = {}
for s in indian_samples:
    decade = (s['age'] // 10) * 10
    age_dist[decade] = age_dist.get(decade, 0) + 1
for d in sorted(age_dist):
    bar = '#' * (age_dist[d] // 20) or '|'
    print(f'  {d:2d}-{d+9:2d}: {age_dist[d]:5d}  {bar}')

males = sum(1 for s in indian_samples if s['gender'] == 'Male')
print(f'\nMale: {males} | Female: {total - males}')
print(f'\n--- Step 2 Complete: {total} images ready ---')


In [ ]:
#@title Step 3: Configure Training

import importlib
import agevision_api.sam.train_config as _tc
importlib.reload(_tc)
from agevision_api.sam.train_config import SAMTrainConfig

# Auto-detect batch size from GPU memory
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
BATCH_SIZE = 4 if gpu_mem >= 40 else 2 if gpu_mem >= 15 else 1

config = SAMTrainConfig(
    data_root=IMAGES_DIR,
    age_csv=csv_path,
    checkpoint_path=CHECKPOINT_PATH,
    output_dir=f'{WORK_DIR}/checkpoints/sam_finetune',
    num_epochs=20,
    batch_size=BATCH_SIZE,
    lr_encoder=5e-5,
    lr_decoder=1e-5,
    lr_discriminator=1e-4,
    lambda_l1=1.0,
    lambda_identity=1.0,
    lambda_age=0.1,
    lambda_adv=0.01,
    lambda_lpips=1.0,
    freeze_decoder=True,
    gradient_checkpointing=True,
    gradient_accumulation_steps=4,
    mixed_precision=True,
    num_workers=2,
    save_every=5,
    log_every=10,
)

eff_batch = config.batch_size * config.gradient_accumulation_steps
print(f'GPU: {torch.cuda.get_device_name(0)} ({gpu_mem:.1f} GB)')
print(f'Dataset: {len(indian_samples)} images')
print(f'Batch: {config.batch_size} x {config.gradient_accumulation_steps} accum = {eff_batch} effective')
print(f'Epochs: {config.num_epochs}')
print(f'LR encoder: {config.lr_encoder}')
print(f'Decoder: FROZEN (encoder-only fine-tuning)')
print(f'Checkpoint: {os.path.basename(config.checkpoint_path)}')
print(f'\n--- Step 3 Complete ---')


In [ ]:
#@title Step 4: START TRAINING

from argparse import Namespace
from torch.amp import GradScaler, autocast
from torch.utils.checkpoint import checkpoint as grad_checkpoint

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s',
                    datefmt='%H:%M:%S', force=True)
logger = logging.getLogger('sam.train')

# ---- Reload modules ----
import agevision_api.sam.dataset as _ds; importlib.reload(_ds)
import agevision_api.sam.losses as _ls; importlib.reload(_ls)
from agevision_api.sam.dataset import create_data_loaders
from agevision_api.sam.losses import SAMTrainingLoss
from agevision_api.sam.models.psp import pSp
from agevision_api.sam.models.stylegan2.model import Discriminator

# ---- Forward with gradient checkpointing ----
def forward_with_checkpointing(net, input_4ch, use_checkpointing=True):
    codes = net.encoder(input_4ch)
    if getattr(net.opts, 'start_from_latent_avg', False):
        codes = codes + net.latent_avg
    elif getattr(net.opts, 'start_from_encoded_w_plus', False):
        if hasattr(net, 'pretrained_encoder') and net.pretrained_encoder is not None:
            with torch.no_grad():
                enc_lat = net.pretrained_encoder(input_4ch[:, :-1, :, :])
                enc_lat = enc_lat + net.latent_avg
            codes = codes + enc_lat
        elif net.latent_avg is not None:
            codes = codes + net.latent_avg
    def _dec(c):
        imgs, _ = net.decoder([c], input_is_latent=True, randomize_noise=False, return_latents=False)
        return imgs
    if use_checkpointing and codes.requires_grad:
        images = grad_checkpoint(_dec, codes, use_reentrant=False)
    else:
        images = _dec(codes)
    return net.face_pool(images)

# ---- Build model ----
device = torch.device('cuda')

def build_opts(ckpt_path, cfg):
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    opts_dict = ckpt.get('opts', {})
    if isinstance(opts_dict, Namespace):
        opts_dict = vars(opts_dict)
    opts_dict.update({
        'checkpoint_path': ckpt_path, 'device': cfg.device,
        'input_nc': cfg.input_nc, 'output_size': cfg.output_size,
        'start_from_latent_avg': cfg.start_from_latent_avg,
        'start_from_encoded_w_plus': cfg.start_from_encoded_w_plus,
    })
    return Namespace(**opts_dict)

logger.info('Loading SAM model...')
opts = build_opts(CHECKPOINT_PATH, config)
opts.device = str(device)
net = pSp(opts).to(device)
net.train()

# Freeze pretrained encoder
if hasattr(net, 'pretrained_encoder') and net.pretrained_encoder is not None:
    for p in net.pretrained_encoder.parameters():
        p.requires_grad = False
    net.pretrained_encoder.eval()
    logger.info('Pretrained encoder frozen')

# Freeze decoder
for p in net.decoder.parameters():
    p.requires_grad = False
net.decoder.eval()
logger.info('Decoder FROZEN (encoder-only fine-tuning)')
trainable_params = list(net.encoder.parameters())

# Discriminator
discriminator = Discriminator(256).to(device)

# Optimizers
opt_gen = torch.optim.Adam(trainable_params, lr=config.lr_encoder, weight_decay=config.weight_decay)
opt_disc = torch.optim.Adam(discriminator.parameters(), lr=config.lr_discriminator, weight_decay=config.weight_decay)

# Data
logger.info('Loading dataset...')
train_loader, val_loader = create_data_loaders(config)
logger.info(f'Train: {len(train_loader)} batches | Val: {len(val_loader)} batches')

# Losses
criterion = SAMTrainingLoss(config).to(device)
if hasattr(net, 'pretrained_encoder') and net.pretrained_encoder is not None:
    criterion.set_identity_encoder(net.pretrained_encoder)
    logger.info('Identity loss: pretrained pSp encoder')

# Mixed precision
use_amp = config.mixed_precision and device.type == 'cuda'
scaler = GradScaler(device=device.type, enabled=use_amp)
accum_steps = config.gradient_accumulation_steps
use_ckpt = config.gradient_checkpointing

allocated = torch.cuda.memory_allocated() / 1e9
total_gpu = torch.cuda.get_device_properties(0).total_memory / 1e9
logger.info(f'VRAM after load: {allocated:.1f}/{total_gpu:.1f} GB | AMP: {use_amp}')

# ---- Save function ----
def save_checkpoint(net, opts, disc, opt_g, opt_d, epoch, cfg):
    os.makedirs(cfg.output_dir, exist_ok=True)
    save_dict = {
        'state_dict': net.state_dict(),
        'opts': vars(opts),
        'discriminator': disc.state_dict(),
        'opt_gen': opt_g.state_dict(),
        'opt_disc': opt_d.state_dict(),
        'epoch': epoch,
    }
    if hasattr(net, 'latent_avg') and net.latent_avg is not None:
        save_dict['latent_avg'] = net.latent_avg
    path = os.path.join(cfg.output_dir, 'sam_indian_finetuned.pt')
    torch.save(save_dict, path)
    # Also copy to /kaggle/working for easy download
    shutil.copy2(path, '/kaggle/working/sam_indian_finetuned.pt')
    logger.info(f'Checkpoint saved (epoch {epoch})')

# ============================================================
#  TRAINING LOOP
# ============================================================
print(f'\n{"="*60}')
print(f'  SAM Fine-tuning: {len(indian_samples)} Indian faces')
print(f'  Epochs: {config.num_epochs} | Batch: {config.batch_size}x{accum_steps}={config.batch_size*accum_steps}')
print(f'  GPU: {torch.cuda.get_device_name(0)} | Decoder: FROZEN')
print(f'{"="*60}\n')

start_time = time.time()
best_val_loss = float('inf')

for epoch in range(1, config.num_epochs + 1):
    epoch_start = time.time()
    net.encoder.train()
    discriminator.train()
    epoch_losses = {'gen': 0, 'disc': 0, 'l1': 0, 'id': 0, 'adv': 0}
    batch_count = 0

    for batch_idx, batch in enumerate(train_loader):
        inp = batch['input'].to(device)
        tgt = batch['target'].to(device)
        ages = batch['real_age'].to(device)

        # Generator step
        with autocast(device_type='cuda', enabled=use_amp):
            gen = forward_with_checkpointing(net, inp, use_checkpointing=use_ckpt)
            d_fake_g = discriminator(gen)
            g_loss, ld = criterion(gen, tgt, target_ages=ages, disc_fake=d_fake_g)
            g_loss = g_loss / accum_steps
        scaler.scale(g_loss).backward()
        if (batch_idx + 1) % accum_steps == 0:
            scaler.step(opt_gen)
            opt_gen.zero_grad(set_to_none=True)

        # Discriminator step
        with autocast(device_type='cuda', enabled=use_amp):
            d_real = discriminator(tgt)
            d_fake = discriminator(gen.detach())
            d_loss = criterion.discriminator_loss(d_real, d_fake) / accum_steps
        scaler.scale(d_loss).backward()
        if (batch_idx + 1) % accum_steps == 0:
            scaler.step(opt_disc)
            opt_disc.zero_grad(set_to_none=True)
            scaler.update()

        del gen, d_fake_g, d_real, d_fake
        torch.cuda.empty_cache()

        epoch_losses['gen'] += ld.get('total', 0)
        epoch_losses['disc'] += d_loss.item()
        epoch_losses['l1'] += ld.get('l1', 0)
        epoch_losses['id'] += ld.get('identity', 0)
        epoch_losses['adv'] += ld.get('adv', 0)
        batch_count += 1

        if (batch_idx + 1) % config.log_every == 0:
            vram = torch.cuda.memory_allocated() / 1e9
            logger.info(
                f'E{epoch}/{config.num_epochs} [{batch_idx+1}/{len(train_loader)}] '
                f'Gen:{ld.get("total",0):.4f} D:{d_loss.item():.4f} '
                f'L1:{ld.get("l1",0):.4f} ID:{ld.get("identity",0):.4f} VRAM:{vram:.1f}GB'
            )

    # Epoch summary
    et = time.time() - epoch_start
    tt = time.time() - start_time
    if batch_count > 0:
        avg = {k: v/batch_count for k,v in epoch_losses.items()}
        logger.info(
            f'Epoch {epoch}/{config.num_epochs} ({et:.0f}s) | '
            f'Gen:{avg["gen"]:.4f} Disc:{avg["disc"]:.4f} '
            f'L1:{avg["l1"]:.4f} ID:{avg["id"]:.4f} | Total:{tt/60:.1f}min'
        )

    # Validation
    if val_loader and epoch % config.save_every == 0:
        net.eval()
        vl_sum, vl_n = 0.0, 0
        with torch.no_grad():
            for vb in val_loader:
                vi = vb['input'].to(device)
                vt = vb['target'].to(device)
                vg = forward_with_checkpointing(net, vi, use_checkpointing=False)
                vl, _ = criterion(vg, vt)
                vl_sum += vl.item(); vl_n += 1
                del vi, vt, vg
        val_loss = vl_sum / max(vl_n, 1)
        logger.info(f'Val Loss: {val_loss:.4f} (best: {best_val_loss:.4f})')
        if val_loss < best_val_loss:
            best_val_loss = val_loss
        net.encoder.train()

    # Save
    if epoch % config.save_every == 0 or epoch == config.num_epochs:
        save_checkpoint(net, opts, discriminator, opt_gen, opt_disc, epoch, config)

total_time = time.time() - start_time
print(f'\n{"="*60}')
print(f'  TRAINING COMPLETE! ({total_time/3600:.1f} hours)')
print(f'  Best val loss: {best_val_loss:.4f}')
print(f'  Checkpoint: /kaggle/working/sam_indian_finetuned.pt')
print(f'{"="*60}')


In [ ]:
#@title Step 5: Quick Visual Test (optional)
import numpy as np
from PIL import Image
from IPython.display import display
import torchvision.transforms as transforms

try:
    finetuned_path = '/kaggle/working/sam_indian_finetuned.pt'
    if not os.path.isfile(finetuned_path):
        print('No checkpoint yet. Run Step 4 first.')
    else:
        net.eval()
        from agevision_api.sam.datasets.augmentations import AgeTransformer

        # Pick a young face
        young = [s for s in indian_samples if 10 <= s['age'] <= 20]
        if not young:
            young = indian_samples[:1]
        test_sample = young[0]

        transform = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3),
        ])

        img = Image.open(f'{IMAGES_DIR}/{test_sample["filename"]}').convert('RGB')
        img_t = transform(img)

        target_ages = [test_sample['age'], 30, 50, 70]
        results = []
        for age in target_ages:
            age_tf = AgeTransformer(target_age=age)
            inp = age_tf(img_t).unsqueeze(0).to(device)
            with torch.no_grad():
                out = forward_with_checkpointing(net, inp, use_checkpointing=False)
            out_img = (out.squeeze().cpu().clamp(-1,1) + 1) / 2
            out_img = transforms.ToPILImage()(out_img)
            results.append((age, out_img))

        # Display grid
        w, h = results[0][1].size
        grid = Image.new('RGB', (w * (len(results)+1), h + 30), 'white')
        orig_resized = img.resize((w, h))
        grid.paste(orig_resized, (0, 30))
        from PIL import ImageDraw
        draw = ImageDraw.Draw(grid)
        draw.text((5, 5), f'Original (age {test_sample["age"]})', fill='black')
        for i, (age, out_img) in enumerate(results):
            grid.paste(out_img, ((i+1)*w, 30))
            draw.text(((i+1)*w + 5, 5), f'Age {age}', fill='black')
        display(grid)
        print('Age progression test complete!')

except Exception as e:
    print(f'Visual test skipped: {e}')


In [ ]:
#@title Step 6: Download Checkpoint

output_path = '/kaggle/working/sam_indian_finetuned.pt'
if os.path.isfile(output_path):
    size = os.path.getsize(output_path) / 1e9
    print(f'Checkpoint: {output_path}')
    print(f'Size: {size:.2f} GB')
    print()
    print('To download:')
    print('  1. Click "Save Version" (top right)')
    print('  2. Save & Run All -> Save')
    print('  3. Go to notebook Output tab')
    print('  4. Download sam_indian_finetuned.pt')
    print()
    print('After download:')
    print('  1. Rename to sam_indian_best.pt')
    print('  2. Copy to agevision_backend/checkpoints/')
    print('  3. Restart Django server')
else:
    print('No checkpoint found. Run Step 4 first.')
